In [1]:
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

C:\Users\zak\Projects\PyCharmProjects\data-science\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_train = kagglehub.dataset_load(adapter=KaggleDatasetAdapter.PANDAS,
                                  handle='house-prices-advanced-regression-techniques', path='train.csv')
df_test = kagglehub.dataset_load(adapter=KaggleDatasetAdapter.PANDAS, handle='house-prices-advanced-regression-techniques', path='test.csv')


In [3]:
from sklearn.model_selection import train_test_split

# Load only the Kaggle 'train.csv'
X = df_train.drop('SalePrice', axis=1)
y = df_train['SalePrice']
# y = np.log1p(df_train['SalePrice'])

I need to treat each end-to-end pipeline as an experiment. The output metrics of the experiment should be known and then tracked and monitored between each run.

Everything should take place within the cross-validation folds, and hopefully there is only one kaggle submit at the end. What you get in the test is what you get in the test.

What do I know at this stage?
- The metric used for the competition "Submissions are evaluated on Root-Mean-Squared-Error (RMSE) between the logarithm of the predicted value and the logarithm of the observed sales price."

In [4]:
from utils.custom_estimators import DropMissingColumns

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor, make_column_selector
from sklearn.model_selection import cross_validate

from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SelectPercentile, mutual_info_regression

In [8]:
keep_numeric = ColumnTransformer(transformers=[
    ('num_filter', 'passthrough', make_column_selector(dtype_include=np.number))
])

# Test different versions of the pipeline and test parameters for each element of the pipeline
#

naive_pipeline = Pipeline(steps=[
    ('keep only numeric features', keep_numeric),
    ('drop missing columns', DropMissingColumns()),

    # Even though there is no missing data left in the training data, some columns in the test dataset have missing values.
    # The fit of the transformers before would not include these as columns to drop
    ('impute_test_missing', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),

    # ('pca', PCA(n_components=0.95)),
    ('feature_selection', SelectPercentile(score_func=mutual_info_regression, percentile=10)),
    ('linear model', LinearRegression()),
])

model = TransformedTargetRegressor(
    naive_pipeline,
    transformer=None,
    func=np.log,
    inverse_func=np.exp
)

results = cross_validate(model, X, y, cv=5, return_train_score=True)
results['train_score'].mean(), results['train_score'].std(), results['test_score'].mean(), results['test_score'].std()


(np.float64(0.6913860687937502),
 np.float64(0.09427453688996479),
 np.float64(0.7155596191399065),
 np.float64(0.1310810626349152))

In [9]:
keep_numeric = ColumnTransformer(transformers=[
    ('num_filter', 'passthrough', make_column_selector(dtype_include=np.number))
])

# Test different versions of the pipeline and test parameters for each element of the pipeline
#

naive_pipeline = Pipeline(steps=[
    ('keep only numeric features', keep_numeric),
    ('drop missing columns', DropMissingColumns()),

    # Even though there is no missing data left in the training data, some columns in the test dataset have missing values.
    # The fit of the transformers before would not include these as columns to drop
    ('impute_test_missing', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),

    # ('pca', PCA(n_components=0.95)),
    ('feature_selection', SelectPercentile(score_func=mutual_info_regression, percentile=10)),
    ('linear model', LinearRegression()),
])

model = TransformedTargetRegressor(
    naive_pipeline,
    transformer=None,
    func=np.log,
    inverse_func=np.exp
)

results = cross_validate(model, X, y, cv=5, return_train_score=True)
results['train_score'].mean(), results['train_score'].std(), results['test_score'].mean(), results['test_score'].std()

(np.float64(0.6605018276689079),
 np.float64(0.1196526763141005),
 np.float64(0.723795591139582),
 np.float64(0.12451365427964162))

In [10]:
from sklearn.model_selection import GridSearchCV

# How many parameters can I try
# How does the linear model compare to a decision tree?
# Can I come up my own utils for solving kaggle problems?

# find another regression problem
# find another notebook and run my own pipeline
# look into decision trees

In [11]:
param_grid = {"regressor__feature_selection__percentile": [1, 3, 10, 30],
              "regressor__impute_test_missing__strategy": ['mean', 'median'],
              "regressor__poly__degree": [1, 2, 3, 4]
              }

In [12]:
from sklearn.model_selection import GridSearchCV

keep_numeric = ColumnTransformer(transformers=[
    ('num_filter', 'passthrough', make_column_selector(dtype_include=np.number))
])

# Test different versions of the pipeline and test parameters for each element of the pipeline

naive_pipeline = Pipeline(steps=[
    ('keep only numeric features', keep_numeric),
    ('drop missing columns', DropMissingColumns()),

    # Even though there is no missing data left in the training data, some columns in the test dataset have missing values.
    # The fit of the transformers before would not include these as columns to drop
    ('impute_test_missing', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(2)),

    # ('pca', PCA(n_components=0.95)),
    ('feature_selection', SelectPercentile(score_func=mutual_info_regression)),
    ('linear model', LinearRegression()),
])

model = TransformedTargetRegressor(
    naive_pipeline,
    transformer=None,
    func=np.log,
    inverse_func=np.exp
)

In [13]:
grid = GridSearchCV(model, param_grid, cv=5, n_jobs=-1)
grid.fit(X, y)

print(f"Best parameters: {grid.best_params_}")
print(f"Best cross-validation score: {grid.best_score_:.3f}")

C:\Users\zak\Projects\PyCharmProjects\data-science\.venv\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [ 6.71629225e-001  6.52256400e-001 -9.97221000e+002             -inf
  6.71629225e-001  6.54772490e-001 -5.62661109e+001              nan
  6.75224377e-001  6.39133546e-001 -5.86673689e+003              nan
  6.75224377e-001  6.35218406e-001 -4.09275081e+003              nan
  7.23439558e-001  6.69941865e-001             -inf -3.72091691e+030
  7.24531728e-001  7.16306234e-001 -4.25332163e+275 -3.94082987e+035
  4.49788886e-001 -1.36711538e+005 -4.68955023e+091 -4.10135969e+007
  4.49788886e-001 -7.76957821e+004 -3.07425674e+070 -7.81414683e+007]
  warnings.warn(
C:\Users\zak\Projects\PyCharmProjects\data-science\.venv\Lib\site-packages\sklearn\model_selection\_search.py:1148: RuntimeWarning: invalid value encountered in subtract
  (array - array_means[:, np.newaxis]) ** 2, axis=1, weights=weights
C:\Users\zak\Pr

Best parameters: {'regressor__feature_selection__percentile': 10, 'regressor__impute_test_missing__strategy': 'median', 'regressor__poly__degree': 1}
Best cross-validation score: 0.725


In [14]:
# 1. Define specific transformers for each data type
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 2. Use ColumnTransformer to apply them to the right columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, make_column_selector(dtype_include=np.number)),
        ('cat', categorical_transformer, make_column_selector(dtype_include=object))
    ])

pipeline_2 = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('pca', PCA(n_components=100)),
        ('poly', )
        # ('feature_selection', SelectPercentile(score_func=mutual_info_regression, percentile=100)),
        ('linear_regression', LinearRegression())
    ],
    verbose=True)

model_2 = TransformedTargetRegressor(
    regressor=pipeline_2,
    transformer=None,
    func=np.log1p,
    inverse_func=np.expm1
)
# Cross-validate the model
from sklearn.model_selection import cross_validate

results = cross_validate(model_2, X, y, return_train_score=True, verbose=True)
-np.log(results['train_score'])

<>:23: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
C:\Users\zak\AppData\Local\Temp\ipykernel_1592\296431050.py:23: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
  ('poly', )


TypeError: 'tuple' object is not callable

In [ ]:
model_2

In [ ]:
submit_to_kaggle(model_2, X, y, df_test, 'submission_3.csv')
# !kaggle competitions submit -c house-prices-advanced-regression-techniques -f submission_3.csv -m "handling categorical data and pca"

In [ ]:
# !kaggle competitions submissions -c house-prices-advanced-regression-techniques

In [ ]:
from kaggle.api.kaggle_api_extended import KaggleApi

# 1. Authenticate
api = KaggleApi()
api.authenticate()

# 2. Define competition and submission details
COMPETITION = 'house-prices-advanced-regression-techniques'


def get_latest_score(competition):
    # Fetch the list of all your submissions
    submissions = api.competition_submissions(competition)

    if submissions:
        latest = submissions[0]
        # Status will be 'pending' while Kaggle is still calculating
        return latest.public_score, latest.date, latest.status
    return None

In [ ]:
def submit_to_kaggle(model, X, y, test_data, file_name) -> None:
    kaggle_model = model.fit(X, y)
    kaggle_predictions = kaggle_model.predict(test_data)
    kaggle_submission_df = pd.DataFrame({
        'Id': test_data['Id'],
        'SalePrice': kaggle_predictions
    })
    kaggle_submission_df.to_csv(file_name, index=False, header=True)

In [ ]:
submit_to_kaggle(model, X, y, df_test, 'submission_2.csv')

In [ ]:
# !kaggle competitions submit -c house-prices-advanced-regression-techniques -f submission_3.csv -m "handling categorical data and pca"

In [ ]:
# !kaggle competitions submissions -c house-prices-advanced-regression-techniques

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import learning_curve
from plotnine import *

# 1. Setup helper to generate learning curve data
def get_learning_curve_df(model, X, y, name):
    train_sizes, train_scores, cv_scores = learning_curve(
        model, X, y, cv=5, scoring='neg_mean_squared_error',
        train_sizes=np.linspace(0.1, 1.0, 10)
    )

    # Calculate mean MSE (negative of neg_mean_squared_error)
    df = pd.DataFrame({
        'Size': train_sizes,
        'Train': -np.mean(train_scores, axis=1),
        'CV': -np.mean(cv_scores, axis=1)
    }).melt(id_vars='Size', var_name='Type', value_name='Error')

    df['Scenario'] = name
    return df

# 2. Generate Synthetic Data (Non-linear relationship)
np.random.seed(42)
X = np.linspace(0, 1, 100).reshape(-1, 1)
y = 0.5 * X**2 + X + 2 + np.random.normal(0, 0.05, (100,))

# Scenario A: High Bias (Underfitting with a simple linear model)
df_bias = get_learning_curve_df(LinearRegression(), X, y, "High Bias (Underfit)")

# Scenario B: High Variance (Overfitting with high-degree polynomial)
X_poly = PolynomialFeatures(degree=15).fit_transform(X)
df_var = get_learning_curve_df(LinearRegression(), X_poly, y, "High Variance (Overfit)")

# 3. Combine and Plot
df_plot = pd.concat([df_bias, df_var])

plot = (
    ggplot(df_plot, aes(x='Size', y='Error', color='Type'))
    + geom_line(size=1.2)
    + geom_point()
    + facet_wrap('~Scenario', scales='free_y')
    + theme_minimal()
    + labs(
        title="Andrew Ng's Model Diagnostics",
        subtitle="Training vs. Cross-Validation Error",
        x="Number of Training Examples (m)",
        y="Error (MSE)"
    )
    + scale_color_manual(values={"Train": "#2c3e50", "CV": "#e74c3c"})
)

plot.draw(show=True)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import learning_curve
from plotnine import *

# 1. Run the Learning Curve on your complex pipeline
# We use neg_root_mean_squared_error (RMSE) as it's easier to interpret
# on the original scale of your target variable.
train_sizes, train_scores, cv_scores = learning_curve(
    estimator=model_2,
    X=X,
    y=y,
    cv=5,
    scoring='neg_root_mean_squared_error',
    train_sizes=np.linspace(0.1, 1.0, 10), # 10 splits from 10% to 100% of data
    n_jobs=-1, # Use all CPU cores since your pipeline is heavy
    verbose=1
)

# 2. Convert negative RMSE to positive RMSE and calculate means
train_scores_mean = -np.mean(train_scores, axis=1)
cv_scores_mean = -np.mean(cv_scores, axis=1)

# 3. Wrangle into a "long" DataFrame for plotnine
df_lc = pd.DataFrame({
    'Training Size (m)': train_sizes,
    'Training Error': train_scores_mean,
    'CV Error': cv_scores_mean
}).melt(
    id_vars='Training Size (m)',
    var_name='Error Type',
    value_name='RMSE'
)



# 4. Plot using plotnine
learning_curve_plot = (
    ggplot(df_lc, aes(x='Training Size (m)', y='RMSE', color='Error Type'))
    + geom_line(size=1.2)
    + geom_point(size=3)
    + theme_minimal()
    + labs(
        title="Learning Curve for Pipeline (PCA + Linear Reg)",
        subtitle="Diagnosing Bias vs. Variance",
        x="Number of Training Examples (m)",
        y="Error (RMSE)"
    )
    + scale_color_manual(values={"Training Error": "#2c3e50", "CV Error": "#e74c3c"})
)

# Explicitly draw the plot so it renders!
learning_curve_plot.draw(show=True)

In [ ]:
import matplotlib.pyplot as plt

x = np.random.rand(50)
y = np.random.rand(50)

plt.plot(x, y)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Generate 1D data (50 points)
x = np.random.rand(50)
y = np.random.rand(50)

# 2. Sort the data (otherwise the line will be a mess)
inds = x.argsort()
x = x[inds]
y = y[inds]

# 3. Use the OO interface
fig, ax = plt.subplots(figsize=(8, 5), facecolor='#f0f0f0') # Set a light background

# 4. Plot with professional styling
ax.plot(x, y, color='#1f77b4', linewidth=2, marker='o', markersize=4, label='Random Trend')

# 5. Clean up the axes
ax.set_title("Cleaned Random Data Plot", fontsize=14, fontweight='bold')
ax.set_xlabel("X Axis (Normalized)")
ax.set_ylabel("Y Axis (Normalized)")
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend()

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Define a layout where the Validation Curve spans the top row,
# and the diagnostics are in a grid below it.
layout = """
    VVV
    LRA
    LRC
"""

fig, ax_dict = plt.subplot_mosaic(layout, figsize=(14, 10))
fig.suptitle("Polynomial Regression Analysis Dashboard", fontsize=16, fontweight='bold')

# Now you access your axes by their string keys!
ax_dict['V'].set_title("Validation Curve (Complexity vs. Error)")
ax_dict['L'].set_title("Learning Curve (Data vs. Error)")
ax_dict['R'].set_title("Residuals Plot")
ax_dict['A'].set_title("Actual vs. Predicted")
ax_dict['C'].set_title("Coefficient Magnitudes")

# Add layout="constrained" alternative for perfect spacing
fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import validation_curve, learning_curve, cross_val_predict
# ... (Assume your Pipeline and model_2 are defined here as you provided) ...

# ==========================================
# PART 1: GATHER THE DIAGNOSTIC DATA
# ==========================================

# 1. Validation Curve Data (Let's test PCA component impact)
pca_range = [10, 50, 100, 150, 200]
train_scores_vc, test_scores_vc = validation_curve(
    model_2, X, y,
    param_name="regressor__pca__n_components",
    param_range=pca_range, cv=5, scoring='neg_mean_squared_error'
)

# 1. Validation Curve Data (Let's test PCA component impact)
poly_range = [1, 2, 3, 4, 5]
train_scores_vc, test_scores_vc = validation_curve(
    model_2, X, y,
    param_name="regressor__pca__n_components",
    param_range=pca_range, cv=5, scoring='neg_mean_squared_error'
)

# 2. Learning Curve Data
train_sizes, train_scores_lc, test_scores_lc = learning_curve(
    model_2, X, y, cv=5, scoring='neg_mean_squared_error'
)

# 3. Predictions for Residuals & Goodness of Fit
y_pred = cross_val_predict(model_2, X, y, cv=5)
residuals = y - y_pred

# 4. Fit the model once on all data to extract final coefficients
model_2.fit(X, y)
# Access the linear regression step inside the pipeline inside the regressor
coefs = model_2.regressor_.named_steps['linear_regression'].coef_

# ==========================================
# PART 2: PLOT THE DASHBOARD
# ==========================================

layout = [
    ['pca_components', 'pca_components', 'pca_components'],
    ['polynomial', 'polynomial', 'polynomial'],
    ['learning',   'residuals',  'actual_pred'],
    ['learning',   'residuals',  'coefficients']
]
fig, ax_dict = plt.subplot_mosaic(layout, figsize=(16, 10))
fig.suptitle("Pipeline Diagnostic Dashboard", fontsize=16, fontweight='bold')

# --- V: PCA Components Validation Curve ---
ax_dict['pca_components'].plot(param_range, -train_scores_vc.mean(axis=1), label='Train Error', marker='o')
ax_dict['pca_components'].plot(param_range, -test_scores_vc.mean(axis=1), label='CV Error', marker='o')
ax_dict['pca_components'].set_title("Validation Curve (PCA Components vs Error)")
ax_dict['pca_components'].set_xlabel("n_components")
ax_dict['pca_components'].set_ylabel("MSE")
ax_dict['pca_components'].legend()

# --- P: Polynomial Validation Curve ---
ax_dict['polynomial'].plot(param_range, -train_scores_vc.mean(axis=1), label='Train Error', marker='o')
ax_dict['polynomial'].plot(param_range, -test_scores_vc.mean(axis=1), label='CV Error', marker='o')
ax_dict['polynomial'].set_title("Validation Curve (PCA Components vs Error)")
ax_dict['polynomial'].set_xlabel("n_components")
ax_dict['polynomial'].set_ylabel("MSE")
ax_dict['polynomial'].legend()

# --- L: Learning Curve ---
ax_dict['L'].plot(train_sizes, -train_scores_lc.mean(axis=1), label='Train Error', marker='o')
ax_dict['L'].plot(train_sizes, -test_scores_lc.mean(axis=1), label='CV Error', marker='o')
ax_dict['L'].set_title("Learning Curve")
ax_dict['L'].set_xlabel("Training Examples")
ax_dict['L'].legend()

# --- R: Residuals Plot ---
ax_dict['R'].scatter(y_pred, residuals, alpha=0.5, edgecolors='k')
ax_dict['R'].axhline(0, color='red', linestyle='--')
ax_dict['R'].set_title("Residuals")
ax_dict['R'].set_xlabel("Predicted Values")
ax_dict['R'].set_ylabel("Residuals (Actual - Predicted)")

# --- A: Actual vs Predicted ---
ax_dict['A'].scatter(y, y_pred, alpha=0.5, edgecolors='k')
# Plot the perfect prediction y=x line
min_val = min(y.min(), y_pred.min())
max_val = max(y.max(), y_pred.max())
ax_dict['A'].plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--')
ax_dict['A'].set_title("Actual vs Predicted")
ax_dict['A'].set_xlabel("Actual Values")
ax_dict['A'].set_ylabel("Predicted Values")

# --- C: Coefficients ---
ax_dict['C'].bar(range(len(coefs)), np.abs(coefs))
ax_dict['C'].set_title("Absolute Coefficient Magnitudes")
ax_dict['C'].set_xlabel("PCA Component Index")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import mplcursors  # Import the library

# 1. Generate some dummy validation curve data
param_range = [10, 50, 100, 150]
cv_errors = [0.8, 0.5, 0.4, 0.9]

fig, ax = plt.subplots(figsize=(8, 5))

# 2. Catch the returned "Artist" (the line).
# Note the comma after 'line'! plot() returns a list of lines.
line, = ax.plot(param_range, cv_errors, marker='o', label='CV Error', color='#ff7f0e')

ax.set_title("Validation Curve with Tooltips")
ax.set_xlabel("n_components")
ax.set_ylabel("MSE")

# 3. Add the interactive cursor
# hover=True makes it appear on mouseover instead of clicking
cursor = mplcursors.cursor(line, hover=True)

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import validation_curve, learning_curve, cross_val_predict

# ==========================================
# 1. GATHER THE DATA (Scikit-Learn)
# ==========================================
# (Assuming model_2, X, and y are already in your notebook memory)

param_range = [10, 50, 100, 150]
# Validation Curve
train_scores_vc, test_scores_vc = validation_curve(
    model_2, X, y, param_name="regressor__pca__n_components",
    param_range=param_range, cv=5, scoring='neg_mean_squared_error'
)

# Learning Curve
train_sizes, train_scores_lc, test_scores_lc = learning_curve(
    model_2, X, y, cv=5, scoring='neg_mean_squared_error'
)

# Predictions & Residuals
y_pred = cross_val_predict(model_2, X, y, cv=5)
residuals = y - y_pred

# Coefficients
model_2.fit(X, y)
coefs = model_2.regressor_.named_steps['linear_regression'].coef_

# Convert negative MSE to positive MSE for easier reading
vc_train_err = -train_scores_vc.mean(axis=1)
vc_test_err = -test_scores_vc.mean(axis=1)
lc_train_err = -train_scores_lc.mean(axis=1)
lc_test_err = -test_scores_lc.mean(axis=1)

# ==========================================
# 2. BUILD THE DASHBOARD GRID
# ==========================================
# We create a 3-row, 2-column grid. The first row spans both columns.
fig = make_subplots(
    rows=3, cols=2,
    specs=[
        [{"colspan": 2}, None],  # Row 1: Validation Curve (Wide)
        [{}, {}],                # Row 2: Learning Curve | Coefficients
        [{}, {}]                 # Row 3: Residuals     | Actual vs Predicted
    ],
    subplot_titles=(
        "Validation Curve (PCA Components vs Error)",
        "Learning Curve", "Absolute Coefficients",
        "Residuals", "Actual vs Predicted"
    ),
    vertical_spacing=0.1, horizontal_spacing=0.1
)

# ==========================================
# 3. ADD THE DATA (TRACES)
# ==========================================

# --- ROW 1: Validation Curve ---
fig.add_trace(go.Scatter(x=param_range, y=vc_train_err, mode='lines+markers', name='Train Error (VC)', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=param_range, y=vc_test_err, mode='lines+markers', name='CV Error (VC)', line=dict(color='orange')), row=1, col=1)

# --- ROW 2, COL 1: Learning Curve ---
fig.add_trace(go.Scatter(x=train_sizes, y=lc_train_err, mode='lines+markers', name='Train Error (LC)', line=dict(color='blue'), showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=train_sizes, y=lc_test_err, mode='lines+markers', name='CV Error (LC)', line=dict(color='orange'), showlegend=False), row=2, col=1)

# --- ROW 2, COL 2: Coefficients ---
fig.add_trace(go.Bar(x=list(range(len(coefs))), y=np.abs(coefs), name='Coeff Magnitude', marker_color='teal'), row=2, col=2)

# --- ROW 3, COL 1: Residuals ---
fig.add_trace(go.Scatter(x=y_pred, y=residuals, mode='markers', name='Residuals', marker=dict(color='red', opacity=0.5)), row=3, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="black", row=3, col=1)

# --- ROW 3, COL 2: Actual vs Predicted ---
fig.add_trace(go.Scatter(x=y, y=y_pred, mode='markers', name='Predictions', marker=dict(color='green', opacity=0.5)), row=3, col=2)
# Add the y=x reference line
min_val, max_val = min(y.min(), y_pred.min()), max(y.max(), y_pred.max())
fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val], mode='lines', line=dict(dash='dash', color='black'), name='Ideal Fit'), row=3, col=2)

# ==========================================
# 4. POLISH & RENDER
# ==========================================
fig.update_layout(
    height=1000,
    width=1000,
    title_text="Pipeline Diagnostic Dashboard",
    hovermode="closest",
    template="plotly_white"  # Gives it a clean, professional look
)

# Render in the notebook
fig.show()